In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType


# ============================================================
# Configuration
# ============================================================
endpoint = "your-url.trycloudflare.com"
BRONZE_PATH = "bronze/idfm/arrets/"
SILVER_PATH = "silver/idfm/arrets/"

In [0]:
import boto3
from botocore.client import Config
import pandas as pd
import io

# Configure boto3 to connect to MinIO
s3_client = boto3.client(
    's3',
    endpoint_url=endpoint,
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    verify=True
)

# Test connection by listing buckets
try:
    buckets = s3_client.list_buckets()
    print(f"Successfully connected to MinIO at {endpoint}")
    print(f"Buckets: {[b['Name'] for b in buckets['Buckets']]}")
except Exception as e:
    print(f"Failed to connect: {e}")

Successfully connected to MinIO at https://assess-player-the-prisoners.trycloudflare.com
Buckets: ['idfm-data']


In [0]:
# List objects in the target path

response = s3_client.list_objects_v2(Bucket='idfm-data', Prefix=BRONZE_PATH, MaxKeys=10)

if 'Contents' in response:
    print(f"\nFound {len(response['Contents'])} objects (showing first 10):")
    parquet_files = [obj['Key'] for obj in response['Contents'] if obj['Key'].endswith('.parquet')]
    
    if parquet_files:
        # Read the first parquet file as an example
        first_file = parquet_files[0]
        print(f"\nReading: {first_file}")
        
        obj = s3_client.get_object(Bucket='idfm-data', Key=first_file)
        parquet_data = obj['Body'].read()
        
        # Read parquet data into pandas DataFrame
        df_pandas = pd.read_parquet(io.BytesIO(parquet_data))
        
        # Convert to Spark DataFrame
        df_bronze = spark.createDataFrame(df_pandas)

        print(df_bronze.printSchema())
        
        print(f"\nDataFrame shape: {df_pandas.shape}")
        
    else:
        print("No parquet files found in the specified path")
else:
    print(f"No objects found with prefix: {BRONZE_PATH}")


Found 2 objects (showing first 10):

Reading: bronze/idfm/arrets/_ingestion_date=2026-08-25/data.parquet
root
 |-- arrid: string (nullable = true)
 |-- arrversion: string (nullable = true)
 |-- arrcreated: string (nullable = true)
 |-- arrchanged: string (nullable = true)
 |-- arrname: string (nullable = true)
 |-- arrtype: string (nullable = true)
 |-- arrxepsg2154: long (nullable = true)
 |-- arryepsg2154: long (nullable = true)
 |-- arrtown: string (nullable = true)
 |-- arrpostalregion: string (nullable = true)
 |-- arraccessibility: string (nullable = true)
 |-- arraudiblesignals: string (nullable = true)
 |-- arrvisualsigns: string (nullable = true)
 |-- arrfarezone: string (nullable = true)
 |-- zdaid: string (nullable = true)
 |-- arrgeopoint: struct (nullable = true)
 |    |-- lat: double (nullable = true)
 |    |-- lon: double (nullable = true)
 |-- _source: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_date: date (nullable =

In [0]:
string_columns = [
    "arrid",
    "arrversion",
    "arrcreated",
    "arrchanged",
    "arrname",
    "arrtype",
    "arrtown",
    "arrpostalregion",
    "arraccessibility",
    "arraudiblesignals",
    "arrvisualsigns",
    "arrfarezone",
    "zdaid",
]

for column in string_columns:
    df_silver = df_bronze.withColumn(
        column,
        F.when(
            F.trim(F.col(column)) == "",
            F.lit(None)
        ).otherwise(
            F.trim(F.col(column))
        )
    )



In [0]:
df_silver = (
    df_silver
    .withColumn(
        "arrcreated_ts",
        F.to_timestamp("arrcreated")
    )
    .withColumn(
        "arrchanged_ts",
        F.to_timestamp("arrchanged")
    )
)

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "latitude",
        F.col("arrgeopoint.lat").cast(DoubleType())
    )
    .withColumn(
        "longitude",
        F.col("arrgeopoint.lon").cast(DoubleType())
    )
)

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "latitude",
        F.when(
            (F.col("latitude") >= -90) &
            (F.col("latitude") <= 90),
            F.col("latitude")
        )
    )
    .withColumn(
        "longitude",
        F.when(
            (F.col("longitude") >= -180) &
            (F.col("longitude") <= 180),
            F.col("longitude")
        )
    )
)

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "x_epsg2154",
        F.col("arrxepsg2154").cast("long")
    )
    .withColumn(
        "y_epsg2154",
        F.col("arryepsg2154").cast("long")
    )
)

In [0]:
categorical_columns = [
    "arrtype",
    "arraccessibility",
    "arraudiblesignals",
    "arrvisualsigns",
    "arrfarezone",
]

for column in categorical_columns:
    df_silver = df_silver.withColumn(
        column,
        F.upper(F.trim(F.col(column)))
    )

In [0]:
df_silver = df_silver.withColumn(
    "arrtown",
    F.initcap(F.trim(F.col("arrtown")))
)

In [0]:
df_silver = df_silver.dropDuplicates(["arrid"])

In [0]:
df_silver = df_silver.filter(
    F.col("arrid").isNotNull()
)

In [0]:
df_silver = df_silver.select(
    # Business identifier
    "arrid",

    # Versioning
    "arrversion",

    # Timestamps
    F.col("arrcreated_ts").alias("created_at"),
    F.col("arrchanged_ts").alias("changed_at"),

    # Stop information
    F.col("arrname").alias("name"),
    F.col("arrtype").alias("type"),
    F.col("arrtown").alias("town"),
    F.col("arrpostalregion").alias("postal_region"),

    # Accessibility
    F.col("arraccessibility").alias("accessibility"),
    F.col("arraudiblesignals").alias("audible_signals"),
    F.col("arrvisualsigns").alias("visual_signs"),

    # Fare
    F.col("arrfarezone").alias("fare_zone"),

    # Geographic information
    F.col("x_epsg2154"),
    F.col("y_epsg2154"),
    F.col("latitude"),
    F.col("longitude"),

    # Geographic / zone identifier
    F.col("zdaid").alias("zda_id"),
)

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)


In [0]:
print("Silver schema:")
df_silver.printSchema()

print(f"Silver records: {df_silver.count()}")

df_silver.show(10, truncate=False)

Silver schema:
root
 |-- arrid: string (nullable = true)
 |-- arrversion: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- changed_at: timestamp (nullable = true)
 |-- name: string (nullable = true)
 |-- type: string (nullable = true)
 |-- town: string (nullable = true)
 |-- postal_region: string (nullable = true)
 |-- accessibility: string (nullable = true)
 |-- audible_signals: string (nullable = true)
 |-- visual_signs: string (nullable = true)
 |-- fare_zone: string (nullable = true)
 |-- x_epsg2154: long (nullable = true)
 |-- y_epsg2154: long (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- zda_id: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = false)

Silver records: 10000
+------+---------------+-------------------+-------------------+--------------------+-----+------------------------+-------------+-------------+---------------+------------+---------+----------+---------

In [0]:
from datetime import datetime
import pyarrow as pa
import pyarrow.parquet as pq

df_silver_pandas = df_silver.toPandas()
# DBTITLE 1,Write to MinIO

# Write to parquet in memory
table = pa.Table.from_pandas(
    df_silver_pandas,
    preserve_index=False
)

# Arrow → Parquet in memory
buffer = io.BytesIO()

pq.write_table(
    table,
    buffer
)

buffer.seek(0)
# Upload to MinIO using boto3
ingestion_date = datetime.now().strftime('%Y-%m-%d')
object_key = f"silver/idfm/arrets/_ingestion_date={ingestion_date}/data.parquet"

try:
    s3_client.put_object(
        Bucket='idfm-data',
        Key=object_key,
        Body=buffer.getvalue()
    )
    print(f"Successfully uploaded to MinIO: {object_key}")
    print(f"Rows written: {len(df_silver_pandas)}")
except Exception as e:
    print(f"Failed to upload: {e}")

    

Successfully uploaded to MinIO: silver/idfm/arrets/_ingestion_date=2026-08-26/data.parquet
Rows written: 10000
